# Clean the satellite tile the same way before LoFTR (iteration 33)

**Idea (from the user).** In iteration 32 only the photo was cleaned (vignette removal + local normalisation). The map tile that LoFTR compares it with was not. Apply the matching treatment to the tile: local normalisation (grey minus local mean over local spread), then the same stretch + CLAHE the photo gets. (The vignette step is lens-specific, so the map gets only the local normalisation.)

**What is measured.** Development photos of 500 m and higher. For each photo the 6 standard queries (headings -5/0/+5 x scales 1.0/1.05) are drawn; each canvas is matched by LoFTR (confidence at least 0.21, RANSAC) with the map tile at the TRUE place. Per photo: the most inliers over the 6 queries. **Control:** the same canvases against a tile 3 km away (a wrong place): the most inliers there are false evidence; a variant that raises true inliers by also raising wrong-place inliers is no use.

Variants: `base` (as the pipeline); `photo cleaned` (vignette + local norm 15 px, iteration 32); `+ tile LN matched` (tile local norm sigma = 15 photo pixels expressed in canvas pixels); `+ tile LN 8`, `+ tile LN 16` (fixed sigma in canvas pixels); `tile LN only` (photo as the pipeline, tile cleaned).

**Kill rule (written before the run).** A variant is useful only if above 800 m the share of photos with at least 8 true-place inliers rises by at least 8 points over `base` AND the share with at least 8 inliers at the wrong place stays at 0 (or not above base), and it does not fall below 800 m. Development photos only; true-place diagnostic like Fable's (it is not the full pipeline).

In [1]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring, fine_matching, flat_field
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

332 photos, heights 100.599998 2016.599976


In [2]:
def edge_cells(image, sigma, cell_valid=None):
    # 8-direction edge strength summed over 4x4-pixel cells, gradient blur = sigma pixels (sigma 1 = the pipeline)
    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), sigma)
    gx = np.zeros_like(blurred); gy = np.zeros_like(blurred)
    gx[:, 1:-1] = blurred[:, 2:] - blurred[:, :-2]; gy[1:-1, :] = blurred[2:, :] - blurred[:-2, :]
    strength = np.hypot(gx, gy); angle = np.arctan2(gy, gx) % np.pi; bin_id = np.minimum((angle / np.pi * settings.N_BINS).astype(int), settings.N_BINS - 1)
    r, c = image.shape[0] // CELL, image.shape[1] // CELL; out = np.zeros((settings.N_BINS, r, c), np.float32)
    for b in range(settings.N_BINS):
        out[b] = (strength * (bin_id == b))[:r * CELL, :c * CELL].reshape(r, CELL, c, CELL).sum(axis=(1, 3))
    return out

def canvas_features(canvas, valid, sigma):
    safe = cv2.erode(valid.astype(np.uint8), np.ones((7, 7), np.uint8)); side = canvas.shape[0] // CELL
    cell_valid = safe.reshape(side, CELL, side, CELL).mean(axis=(1, 3)) >= 0.8
    return normalise_cells(edge_cells(canvas, sigma), cell_valid) * cell_valid, cell_valid

def window_pixels(level_image, top, left, cells_wide):
    size = cells_wide * CELL; r0, c0 = top * CELL, left * CELL; piece = np.zeros((size, size), np.float32); h, w = level_image.shape
    a0, a1 = max(r0, 0), min(r0 + size, h); b0, b1 = max(c0, 0), min(c0 + size, w)
    if a1 > a0 and b1 > b0: piece[a0 - r0:a1 - r0, b0 - c0:b1 - c0] = np.asarray(level_image[a0:a1, b0:b1], np.float32) / 255.0
    return piece

def map_features(piece, sigma):
    feats = edge_cells(piece, sigma); has = (piece[:feats.shape[1] * CELL, :feats.shape[2] * CELL].reshape(feats.shape[1], CELL, feats.shape[2], CELL).min(axis=(1, 3)) > 0)
    has = cv2.erode(has.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    feats = normalise_cells(feats, has) * has; return feats, has

def hsv_picture(feats):
    # hue = dominant edge direction, brightness = total strength (99th percentile = full)
    total = feats.sum(axis=0); hue = feats.argmax(axis=0) / settings.N_BINS; val = np.clip(total / (np.percentile(total, 99) + 1e-6), 0, 1)
    rgb = np.zeros(total.shape + (3,), np.float32)
    for i in range(total.shape[0]):
        for j in range(total.shape[1]): rgb[i, j] = colorsys.hsv_to_rgb(hue[i, j], 0.9, val[i, j])
    return rgb

In [3]:
import matplotlib; matplotlib.use("Agg")
loftr = fine_matching.load_loftr(os.path.join(os.environ.get("EF_MODELS", os.path.join(os.path.abspath(HEADLESS), "models")), settings.LOFTR_FILE))
ORIGINAL_NORMALISE = photo.normalise_frame
def with_settings(flat=None, degree=3, local=None):
    settings.FLAT_FIELD_FROM_HEIGHT_M = flat; settings.FLAT_FIELD_DEGREE = degree; settings.LOCAL_NORM_SIGMA = local
def clean_tile(tile, sigma):
    has = tile > 0; fill = float(tile[has].mean()) if has.any() else 0.5
    filled = np.where(has, tile, fill).astype(np.float32); normal = flat_field.local_normalise(filled, sigma)
    out = ORIGINAL_NORMALISE(normal)                     # the same stretch + CLAHE the photo gets
    return np.where(has, out, float(out[has].mean()) if has.any() else 0.5).astype(np.float32)
def best_inliers(canvases, tile):
    best = 0
    for canvas_picture in canvases:
        pa, pb, conf = fine_matching.loftr_matches(loftr, canvas_picture, tile); keep = conf >= settings.LOFTR_CONFIDENCE
        matrix, inl = fine_matching.fit_similarity(pa[keep], pb[keep]); best = max(best, int(inl.sum()) if matrix is not None else 0)
    return best
VARIANTS = [("base", dict(), None), ("photo cleaned", dict(clean=True), None), ("photo cleaned + tile LN matched", dict(clean=True), "matched"),
            ("photo cleaned + tile LN 8", dict(clean=True), 8.0), ("photo cleaned + tile LN 16", dict(clean=True), 16.0), ("tile LN only (matched)", dict(), "matched")]
query_list = queries.query_list(); records = []
for index, meta in enumerate(rows):
    if float(meta["alt"]) < 500: continue
    camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"]); level = level_for_height(pose["height"], camera); gsd = level_gsd(level)
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); photo_gsd = pose["height"] / camera["fx"]
    base_true = map_piece(store.levels[level], level, truth[0], truth[1], TILE)
    wrong_xy = None
    for dx in (3000.0, -3000.0, 0.0):
        for dy in ((0.0, 3000.0, -3000.0) if dx == 0.0 else (0.0,)):
            cand = map_piece(store.levels[level], level, truth[0] + dx, truth[1] + dy, TILE)
            if wrong_xy is None and (cand > 0).mean() > 0.9: wrong_xy = (truth[0] + dx, truth[1] + dy); base_wrong = cand
    if wrong_xy is None: continue
    pictures = {}
    for clean in (False, True):
        if clean: with_settings(0, 3, 15.0)
        else: with_settings(None)
        pictures[clean] = photo.prepare_photo(path, pose, camera, level)
    with_settings(None); canvases = {}
    for clean in (False, True):
        canvases[clean] = [fine_matching.grey_canvas(*photo.draw_canvas(pictures[clean], pose, camera, level, q["heading"], q["scale_factor"])) for q in query_list]
    for name, kw, tile_mode in VARIANTS:
        cs = canvases[kw.get("clean", False)]
        sigma_tile = None if tile_mode is None else (max(3.0, 15.0 * photo_gsd / gsd) if tile_mode == "matched" else tile_mode)
        tt = base_true if sigma_tile is None else clean_tile(base_true, sigma_tile); tw = base_wrong if sigma_tile is None else clean_tile(base_wrong, sigma_tile)
        records.append(dict(photo=index, height=pose["height"], variant=name, true_inl=best_inliers(cs, tt), wrong_inl=best_inliers(cs, tw)))
    if index % 20 == 0: print(index, flush=True)
tab = pd.DataFrame(records); tab.to_csv(os.path.join(OUT, "loftr_map_cleaning.csv"), index=False)
tab["band"] = pd.cut(tab.height, [500, 800, 1100, 1400, 2100]); order = [v[0] for v in VARIANTS]
summ = lambda g: pd.Series(dict(n=len(g), true_ge8=(g.true_inl >= 8).mean(), true_median=g.true_inl.median(), wrong_ge8=(g.wrong_inl >= 8).mean(), wrong_ge4=(g.wrong_inl >= 4).mean()))
print("by band"); print(tab.groupby(["band", "variant"], observed=True).apply(summ).round(2).unstack("variant").swaplevel(axis=1)[order].sort_index(axis=1, level=0, sort_remaining=False)[["true_ge8", "wrong_ge8"]].to_string() if False else "")
for col in ["true_ge8", "wrong_ge8", "wrong_ge4", "true_median"]:
    print(col); print(tab.groupby(["band", "variant"], observed=True).apply(summ)[col].unstack("variant")[order].round(2).to_string())
hi_ = tab[tab.height >= 800]; print("above 800 m"); print(hi_.groupby("variant").apply(summ).loc[order].round(3).to_string())

/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


20


40


60


140


160


... (4 outputs left out)

280


300


320


by band

true_ge8
variant       base  photo cleaned  photo cleaned + tile LN matched  photo cleaned + tile LN 8  photo cleaned + tile LN 16  tile LN only (matched)
band                                                                                                                                             
(500, 800]    1.00           0.96                             0.97                       0.96                        0.94                    0.94
(800, 1100]   0.60           0.57                             0.57                       0.57                        0.60                    0.51
(1100, 1400]  0.60           0.70                             0.75                       0.75                        0.78                    0.52
(1400, 2100]  0.18           0.19                             0.18                       0.18                        0.18                    0.07
wrong_ge8
variant       base  photo cleaned  photo cleaned + tile LN matched  photo cleaned + tile LN 8  p